# Attributing research-agent errors: scrape vs model

A research agent answers a question by scraping a page and reasoning over the result. When the answer
is wrong, the cause is one of two things: the scrape did not deliver the fact (the model never saw it),
or the model had the fact and still got it wrong. This notebook separates the two on a small labelled
set — scrape each source two ways, then check whether the ground-truth fact is present in the context.

Runs against the Firecrawl cloud API. Provide a key via `FIRECRAWL_API_KEY` or a `.fc-key` file in this
folder (the setup cell loads either).

## Setup

Five questions, each with a known answer on a page that is hard to scrape (JavaScript-rendered, or
behind anti-bot). The agent loop is: scrape the source, then answer from what was scraped. We hold the
question and the model fixed and vary only the scraper:

- naive HTTP — a plain `GET` with tags stripped.
- Firecrawl — `/v2/scrape`.

In [1]:
// setup — load the API key (env var OR a .fc-key file in this folder), then define the scrapers
async function loadKey(): Promise<string> {
  const env = Deno.env.get("FIRECRAWL_API_KEY");
  if (env && env.trim()) return env.trim();
  for (const p of ["./.fc-key", "../.fc-key"]) {
    try { const k = (await Deno.readTextFile(p)).trim(); if (k) return k; } catch { /* next */ }
  }
  return "";
}
const KEY = await loadKey();
const UA = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36";
const auth = () => ({ "Content-Type": "application/json", Authorization: `Bearer ${KEY}` });

async function naiveScrape(url: string) {
  try {
    const r = await fetch(url, { headers: { "User-Agent": UA, Accept: "text/html" }, signal: AbortSignal.timeout(20000) });
    const html = await r.text();
    const text = html.replace(/<script[\s\S]*?<\/script>/gi, " ").replace(/<style[\s\S]*?<\/style>/gi, " ")
      .replace(/<[^>]+>/g, " ").replace(/\s+/g, " ").trim();
    return { status: r.status as number | string, text };
  } catch { return { status: "ERR" as number | string, text: "" }; }
}
// Firecrawl -> clean markdown. Hard anti-bot pages can return a transient miss, so retry.
async function fireScrape(url: string, tries = 3) {
  for (let attempt = 1; attempt <= tries; attempt++) {
    try {
      const r = await fetch("https://api.firecrawl.dev/v2/scrape", { method: "POST", headers: auth(),
        body: JSON.stringify({ url, formats: ["markdown"], onlyMainContent: true, timeout: 50000 }), signal: AbortSignal.timeout(80000) });
      const j = await r.json();
      const text = j?.data?.markdown ?? "";
      if (j.success === true && text.length > 0) return { status: j?.data?.metadata?.statusCode ?? 200, text };
    } catch { /* retry */ }
    if (attempt < tries) await new Promise((r) => setTimeout(r, 1500));
  }
  return { status: "FAIL" as number | string, text: "" };
}
async function agentAnswer(url: string, question: string) {
  const r = await fetch("https://api.firecrawl.dev/v2/scrape", { method: "POST", headers: auth(),
    body: JSON.stringify({ url, formats: [{ type: "json",
      prompt: `${question} Answer in as few words as possible using only the page.`,
      schema: { type: "object", properties: { answer: { type: "string" } }, required: ["answer"] } }], timeout: 60000 }),
    signal: AbortSignal.timeout(90000) });
  const j = await r.json();
  return String(j?.data?.json?.answer ?? "(error)").trim();
}
const has = (t: string, n: RegExp) => n.test(t);
const pad = (s: unknown, n: number) => { const x = String(s); return x.length >= n ? x.slice(0, n) : x + " ".repeat(n - x.length); };
const reason = (s: number | string) => s === 403 || s === 401 ? "blocked (anti-bot)"
  : (typeof s === "number" && s >= 200 && s < 300 ? "silent 200 (JS shell)" : `transport error (${s})`);

const QUESTIONS = [
  { question: "Which physicist is quoted on this page?", source: "https://quotes.toscrape.com/js/", needle: /einstein/i, truth: "Albert Einstein" },
  { question: "In which city is OpenAI headquartered?", source: "https://www.crunchbase.com/organization/openai", needle: /san francisco/i, truth: "San Francisco" },
  { question: "In which city is Shopify headquartered?", source: "https://www.crunchbase.com/organization/shopify", needle: /ottawa/i, truth: "Ottawa" },
  { question: "In which city is Spotify headquartered?", source: "https://www.crunchbase.com/organization/spotify", needle: /stockholm/i, truth: "Stockholm" },
  { question: "What kind of software is Notion, per G2?", source: "https://www.g2.com/products/notion/reviews", needle: /note-taking|productivity|knowledge|project management|collaboration|workspace/i, truth: "productivity / note-taking" },
];
console.log(KEY ? `API key loaded. ${QUESTIONS.length} questions, two scrapers ready.`
                : "⚠️  NO API KEY — set FIRECRAWL_API_KEY or put your key in a .fc-key file in this folder, then re-run.");

API key loaded. 5 questions, two scrapers ready.


## naive HTTP vs Firecrawl on one source

The naive `GET` returns a 403 anti-bot page (a few hundred characters); `/v2/scrape` returns the
profile.

In [2]:
// one source, both scrapers
const demo = QUESTIONS[1].source;
const n0 = await naiveScrape(demo);
const f0 = await fireScrape(demo);
console.log(`source: ${demo}\n`);
console.log(`naive HTTP   status ${n0.status}, ${n0.text.length} chars`);
console.log(`  "${n0.text.slice(0, 130).trim()}…"\n`);
console.log(`Firecrawl  status ${f0.status}, ${f0.text.length} chars`);
console.log(`  "${f0.text.replace(/\n/g, " ").slice(0, 130).trim()}…"`);

source: https://www.crunchbase.com/organization/openai

naive HTTP   status 403, 797 chars
  "Attention Required! | Cloudflare Please enable cookies. Sorry, you have been blocked You are unable to access crunchbase.com Why h…"

Firecrawl  status 200, 47411 chars
  "[Crunchbase](https://www.crunchbase.com/)  Resources  Advanced Search  [Start Free Trial](https://www.crunchbase.com/buy/select-pr…"


## Attribution: scrape failure vs model failure

When the agent is wrong, two checks decide why, and they point at different fixes:

| fact present in context? | answer correct? | verdict | fix |
|:---:|:---:|---|---|
| no  | — | **scrape failure** | the scraper (render / anti-bot) |
| yes | no  | **model failure** | the prompt / model |
| yes | yes | correct | — |

A model can only use a fact that reached the context, so fact-absent implies a scrape failure. Routing
each error to the layer that owns it avoids optimising the wrong one: a better prompt cannot recover a
page that never loaded, and a better scraper cannot fix a reasoning error.

Each question gets one verdict per scraper, then a tally.

In [3]:
// one verdict per question, per scraper — the attribution
let naiveScrapeFails = 0, fireCorrect = 0, fireModelFails = 0, fireScrapeFails = 0;
console.log(pad("question", 40), pad("naive HTTP", 32), "Firecrawl");
console.log("-".repeat(108));
for (const q of QUESTIONS) {
  const n = await naiveScrape(q.source);
  const f = await fireScrape(q.source);
  let naiveVerdict: string;
  if (has(n.text, q.needle)) naiveVerdict = "correct";
  else { naiveVerdict = `SCRAPE failure · ${reason(n.status)}`; naiveScrapeFails++; }
  let fireVerdict: string;
  if (!has(f.text, q.needle)) { fireVerdict = `SCRAPE failure · ${reason(f.status)}`; fireScrapeFails++; }
  else {
    const a = await agentAnswer(q.source, q.question);
    if (q.needle.test(a)) { fireVerdict = `correct · "${a}"`; fireCorrect++; }
    else { fireVerdict = `MODEL failure · got "${a}"`; fireModelFails++; }
  }
  console.log(pad(q.question, 40), pad(naiveVerdict, 32), fireVerdict);
}
console.log("-".repeat(108));
const N = QUESTIONS.length;
console.log(`\nNaive HTTP    agent ${N - naiveScrapeFails}/${N}   ·   ${naiveScrapeFails} SCRAPE failures, 0 model failures`);
console.log(`Firecrawl   agent ${fireCorrect}/${N}   ·   ${fireScrapeFails} scrape failures, ${fireModelFails} model failures, ${fireCorrect} correct`);
console.log(`\nEvery naive error attributes to the SCRAPE — the fact never reached the model. Fix the engine`);
console.log(`(Firecrawl) and the same agent recovers. The model was never the bottleneck.`);

question                                 naive HTTP                       Firecrawl
------------------------------------------------------------------------------------------------------------
Which physicist is quoted on this page?  SCRAPE failure · silent 200 (JS  correct · "Albert Einstein"
In which city is OpenAI headquartered?   SCRAPE failure · blocked (anti-b correct · "San Francisco"
In which city is Shopify headquartered?  SCRAPE failure · blocked (anti-b correct · "Ottawa, Ontario, Canada"
In which city is Spotify headquartered?  SCRAPE failure · blocked (anti-b correct · "Stockholm"
What kind of software is Notion, per G2? SCRAPE failure · blocked (anti-b correct · "AI workspace for project management and collaboration."
------------------------------------------------------------------------------------------------------------

Naive HTTP    agent 0/5   ·   5 SCRAPE failures, 0 model failures
Firecrawl   agent 5/5   ·   0 scrape failures, 0 model failures, 5 correct

Every 

## Result

naive 0/5, Firecrawl 5/5. Every naive failure is a scrape failure — the fact was absent from the
context — and none is a model failure. With the context recovered, the same agent answers all five.

Companion to Part 1: [01-detecting-silent-200s](./01-detecting-silent-200s.ipynb) detects a 200 whose
body is junk; here a stronger scrape recovers the page so the agent can use it.